In [1]:
import sys
sys.path.append("../../src")
import numpy as np
import pandas as pd
import torch
import matplotlib.pylab as plt
from synthetic_observations import Observations
from gaussian_synthetic_observations import Gaussian_Observations
from transformer import *
from spectrum_lsf import Score_Likelihood
from score_models import ScoreModel
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from template import Template
from sbart_rv_finder import RV_Retrieval
from mala import MALA
# from mala import MALAnew
import h5py
from torch.autograd.functional import jacobian


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
change_res=True

# Create Observations

In [4]:
snr = 25
nspec = 2
B = 1
change_res=True
obs = Observations(i=6,SNR=snr,filepath='../../data/SPIRou20_val.df',N=nspec,seed=0,wfile='../../data/SPIRou_wavelength_solution.fits')
synthetic_spectra, uncertainty = obs.make_observations(func='connors',add_RV=True,change_res=change_res,filepath='../../data/SPIRou_wavelength_solution.fits')
synthetic_spectra, uncertainty = obs.post_process()
non_ones = torch.where(obs.padded_wgrid!=1)[0]

# Template

In [5]:
temp = Template(synthetic_spectra, obs.berv, obs.inst_wgrid, obs.wgrid)
template = temp.make_template(func='scipy')
sbart = RV_Retrieval(snr,template,obs.wgrid,obs.inst_wgrid,nspec)
templatervs_init, tempuncs = sbart.find_dv(synthetic_spectra[:,:].cpu(),uncertainty.cpu()[:,:],obs.berv[:].cpu(),func='connors')

# Spectrum Sample

In [6]:
bervs_to_send = obs.berv.unsqueeze(0).expand(B, nspec)
planetrv_forA = torch.tensor(templatervs_init).to(device).unsqueeze(0)
planetrv_for_spectrum_sample = torch.tensor(templatervs_init).to(device).unsqueeze(0).expand(B, nspec)
# planetrv_for_spectrum_sample = obs.planet.clone().unsqueeze(0).expand(B, nspec)

In [7]:
list_AtA = []
for planet_chunk, berv_chunk in zip(obs.planet, obs.berv):
    planetrv_forA = torch.as_tensor(planet_chunk, device=device).unsqueeze(0).unsqueeze(0)
    bervy = berv_chunk.unsqueeze(0).unsqueeze(0)

    def f_wrapped(x):
        return forward_model(x, obs.wgrid, obs.inst_wgrid, bervy, planetrv_forA, change_res)

    x = obs.training[:, :, non_ones]
    A_full = jacobian(f_wrapped, x, create_graph=False)
    A = A_full[0, :, :, 0, 0, :]                 # [chunk, L, L]
    chunk_AtA = torch.matmul(A, A.transpose(-1, -2))   # [chunk, L, L]

    list_AtA.append(chunk_AtA)
    del A_full, A, chunk_AtA
    torch.cuda.empty_cache()

AtA = torch.cat(list_AtA)
list_AtA = []

In [8]:
# AtA = torch.load("AtA.pt")

In [9]:
checkpoints_directory = "../../trained_models/b16nf64ch2_spirfull"
model = ScoreModel(checkpoints_directory=checkpoints_directory, device=device)

Loaded checkpoint 402 of b16nf64ch2_spirfull
Using the Variance Preserving SDE


In [10]:
# def schedule_steps(snr):
#     snr_low, steps_low = 10, 3000
#     snr_high, steps_high = 200, 10000

#     # linear interpolation
#     steps = steps_low + (steps_high - steps_low) * (snr - snr_low) / (snr_high - snr_low)
#     return int(steps)

In [11]:
LSF = Score_Likelihood(Y=synthetic_spectra, V=planetrv_for_spectrum_sample, sig_n=uncertainty,
                                    spec_wgrid=obs.wgrid, inst_wgrid=obs.inst_wgrid, non_ones=non_ones, SNR=snr,
                                    berv=bervs_to_send, beta_min=1e-2, beta_max=20, AtA=AtA.unsqueeze(0), change_res=change_res)
dimensions = [1, len(obs.padded_wgrid)]

posterior_samples = model.sample(shape=[B, *dimensions], steps=3000, likelihood_score_fn=LSF,sampling_order=1.0)

First order Sampling from the posterior | t = 0.9 | sigma = 1.0e+00| scale ~ 8.3e-01:   8%|▊         | 252/3000 [00:19<03:27, 13.22it/s]


KeyboardInterrupt: 

In [ ]:
temp_res = torch.sqrt(torch.mean((obs.planet.cpu()[:] - templatervs_init)**2))
tempmeanzscore = torch.mean((obs.planet.cpu()[:] - templatervs_init)/torch.tensor(tempuncs))
tempstdzscore = torch.std((obs.planet.cpu()[:] - templatervs_init)/torch.tensor(tempuncs))
print(temp_res, tempmeanzscore,tempstdzscore)

In [ ]:
mala = MALA(synthetic_spectra[:,:],uncertainty[:,:],obs.berv[:],snr,obs.inst_wgrid,obs.wgrid)
load_post_sample = posterior_samples[:,:,non_ones]
samples_int, accepted_int = mala.find_rv(planetrv_for_spectrum_sample,load_post_sample,500)
postrvs = torch.mean(samples_int[100:,0],axis=0).cpu()
uncs = torch.std(samples_int[100:,0],axis=0).cpu()
post_res = torch.sqrt(torch.mean((obs.planet.cpu()[:] - postrvs)**2))
postmeanzscore = torch.mean((obs.planet.cpu()[:] - postrvs)/uncs)
poststdzscore = torch.std((obs.planet.cpu()[:] - postrvs)/uncs)
print(post_res, postmeanzscore,poststdzscore)

In [ ]:
plt.plot(posterior_samples[0,0].cpu())
plt.plot(obs.training[0,0].cpu())

In [ ]:
residuals = posterior_samples[0,0]-obs.training[0,0]
temp_res = template - obs.training[0,0][non_ones]

In [ ]:
start = int(0.05 * len(obs.training[0,0][non_ones]))
end = int(0.95 * len(obs.training[0,0][non_ones]))

In [ ]:
plt.figure(figsize=(15,5))

plt.scatter(obs.padded_wgrid[non_ones][start:end].cpu(), temp_res[start:end].cpu(),c="r",alpha=0.5)
plt.scatter(obs.padded_wgrid[non_ones][start:end].cpu(), residuals[non_ones][start:end].cpu(),c="b",alpha=0.5)
plt.hlines(0,obs.padded_wgrid[non_ones][start:end][0].cpu(),obs.padded_wgrid[non_ones][start:end][-1].cpu(),color="k")
# plt.ylim(-0.01,0.01)
# plt.xlim(1301,1305)

# Gibbs MALA

In [ ]:
%reload_ext autoreload

In [ ]:
obs_eval = Observations(i=6,SNR=snr,filepath='../../data/SPIRou20_val.df',N=100,seed=2,wfile='../../data/SPIRou_wavelength_solution.fits')
synthetic_spectra_eval, uncertainty_eval = obs_eval.make_observations(func='connors',add_RV=True,change_res=change_res,filepath='../../data/SPIRou_wavelength_solution.fits')
synthetic_spectra_eval, uncertainty_eval = obs_eval.post_process()
non_ones = torch.where(obs_eval.padded_wgrid!=1)[0]

In [ ]:
sbart_eval = RV_Retrieval(snr,template,obs_eval.wgrid,obs_eval.inst_wgrid,nspec)
templatervs_eval, tempuncs = sbart_eval.find_dv(synthetic_spectra_eval[:,:].cpu(),uncertainty_eval.cpu()[:,:],obs_eval.berv[:].cpu(),func='connors')

In [ ]:
sbart_int = RV_Retrieval(snr, obs_eval.training[:, :, non_ones], obs_eval.wgrid, obs_eval.inst_wgrid, 100, type='sample')
int_rv, int_unc = sbart_int.find_dv(synthetic_spectra_eval.cpu(), uncertainty_eval.cpu(), obs_eval.berv.cpu(), func='connors')

In [ ]:
non_ones.shape

In [ ]:
mala = MALA(synthetic_spectra_eval[:,:],uncertainty_eval[:,:],obs_eval.berv[:],snr,obs_eval.inst_wgrid,obs_eval.wgrid)
init_rvs = torch.tensor(templatervs_eval).to(DEVICE).unsqueeze(0).expand(1,100)
# init_rvs = obs_eval.planet.unsqueeze(0).expand(1,100)
# load_post_sample = loaded_post[-1,0,0,non_ones].unsqueeze(0).unsqueeze(0)
load_post_sample = posterior_samples [:,:,non_ones]
samples_int, accepted_int = mala.find_rv(init_rvs,load_post_sample,500)

In [ ]:
temp_res = torch.sqrt(torch.mean((obs_eval.planet.cpu()[:] - templatervs_eval)**2))
tempmeanzscore = torch.mean((obs_eval.planet.cpu()[:] - templatervs_eval)/torch.tensor(tempuncs))
tempstdzscore = torch.std((obs_eval.planet.cpu()[:] - templatervs_eval)/torch.tensor(tempuncs))
print(temp_res, tempmeanzscore,tempstdzscore)

In [ ]:
int_res = torch.sqrt(torch.mean((obs_eval.planet.cpu()[:] - int_rv)**2))
intmeanzscore = torch.mean((obs_eval.planet.cpu()[:] - int_rv)/torch.tensor(int_unc))
intstdzscore = torch.std((obs_eval.planet.cpu()[:] - int_rv)/torch.tensor(int_unc))
print(int_res, intmeanzscore,intstdzscore)

In [ ]:
postrvs = torch.mean(samples_int[100:,0],axis=0).cpu()
uncs = torch.std(samples_int[100:,0],axis=0).cpu()
post_res = torch.sqrt(torch.mean((obs_eval.planet.cpu()[:] - postrvs)**2))
postmeanzscore = torch.mean((obs_eval.planet.cpu()[:] - postrvs)/uncs)
poststdzscore = torch.std((obs_eval.planet.cpu()[:] - postrvs)/uncs)
print(post_res, postmeanzscore,poststdzscore)

In [ ]:
torch.mean(samples_int[100:], dim=0).shape

In [ ]:
planetrv_for_spectrum_sample.shape

In [ ]:
plt.plot(samples_int[:,0,0].cpu())
plt.plot(samples_int[:,0,4].cpu())
# plt.plot(samples_int[:,0,10].cpu())

In [ ]:
from scipy.stats import norm

fix, axs = plt.subplots(5,3,figsize=(10,10))
axs = axs.flatten()  # Flatten to easily index from 0 to 8

for i in range(15):
    n, bins,_ = axs[i].hist(samples_int[100:,:,i].cpu().flatten(),density=True,alpha=0.6)
    axs[i].vlines(obs_eval.planet[i].cpu(),0,max(n)+0.001,color="k",lw=2)
    axs[i].vlines(templatervs_eval[i],0,max(n)+0.001,color="orange",lw=2)
    axs[i].vlines(int_rv[i],0,max(n)+0.001,color="green",lw=2)

    # Gaussian curve
    x = np.linspace(bins[0], bins[-1], 200)  # smooth x-axis
    # y = norm.pdf(x,templatervs_eval[i], tempuncs[i])  # PDF values
    # axs[i].plot(x, y, 'orange', linewidth=2)

    y = norm.pdf(x,int_rv[i], int_unc[i])  # PDF values
    axs[i].plot(x, y, 'green', linewidth=2)
    axs[i].set_xlim(-30,30)

    